In [1]:
import pandas as pd
import numpy as np
import gradio as gr
import tempfile
import os
from datetime import datetime

# get data from data correction - google docs
def get_data_correction():
    sheet_id = "1XuTFEkznzX6QAr4ZIk27vVSpTpbWtaU2"
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

    df = pd.read_csv(url)
    df['Participant ID'] = df['Participant ID'].str.strip()
    df['Form'] = df['Form'].str.strip()

    return df

# =========================
# =========================
# Main Processing Function A1D1
# =========================
# =========================
def process_files_a1d1(a1_file, d1_file):
    a1_section_mapping = {
        'A1'    : 'ID',
        'A1manual' : 'ID Manual',
        'A4/1'  : 'B.Inclusion Criteria-A4/1',
        'A4/2'  : 'C.Demographic Information-A4/2',
        'A4/3'  : 'D.Animal Contact-A4/3',
        'A4/4'  : 'E.Mosquito Activity Information-A4/4',
        'A4/5'  : 'F.General Knowledge-A4/5',
        'A4/6'  : 'G.Child Information-A4/6',
        'A4/7'  : 'H.Chile Vaccination Record-A4/7',
        'A4/8'  : 'I.Child Medical History-A4/8',
        'A4/9'  : 'J.Physical Examination and Neurological Assessment-A4/9',
        'ID'    : 'A1'
    }

    if a1_file is None or d1_file is None:
        return "❌ Please upload both A1 and D1 files", None, None, None, None, None, None, None

    try:
        # =========================
        # Load files (HF compatible)
        # =========================
        a1_df = pd.read_csv(a1_file, low_memory=False)
        d1_df = pd.read_csv(d1_file, low_memory=False)

        # =========================
        # A1 processing
        # =========================
        a1_df['SubmissionDate'] = pd.to_datetime(a1_df['SubmissionDate'], errors='coerce')
        a1_df.sort_values(by='SubmissionDate', ascending=False, inplace=True)

        A4_cols = [f'A4/{i}' for i in range(1,10)]
        A_section = ['SubmissionDate', 'A1', 'A1manual'] + A4_cols
        a1_df = a1_df[A_section]

        a1_df['A1'] = a1_df['A1'].fillna(a1_df['A1manual'])
        a1_df[A4_cols] = a1_df[A4_cols].apply(pd.to_numeric, errors='coerce')

        # =========================
        # D1 processing
        # =========================
        d1_df['A1'] = d1_df['A1'].fillna(d1_df['A1manual'])

        useful_column = ['SubmissionDate','A1', 'A1manual', 'SubmitterID', 'SubmitterName', 'Edits']
        d1_df = d1_df[useful_column]

        # =========================
        # Main Logic
        # =========================
        result_dict = {}

        for id in d1_df['A1'].dropna().unique():
            tmp_df = a1_df[a1_df['A1'] == id]

            if tmp_df.empty:
                continue

            condition_dict = {
                col: int(tmp_df[col].sum(skipna=True))
                for col in A4_cols
            }

            condition_dict['Record in A1'] = tmp_df.shape[0]

            if not all(condition_dict[col] == 1 for col in A4_cols):
                result_dict[id] = condition_dict

        # =========================
        # Convert to DataFrame
        # =========================
        df = pd.DataFrame.from_dict(result_dict, orient='index').reset_index()
        df = df.rename(columns={'index':'A1'})

        if df.empty:
            return "✅ NO ISSUES FOUND", None, None, None, None, None, None, None

        # =========================
        # Add remarks
        # =========================
        df['A4_max'] = df[A4_cols].max(axis=1)

        df['Remark'] = np.select(
            [
                df['A4_max'] == 0,
                df['A4_max'] == 2,
                df['A4_max'] == 3,
                df['A4_max'] > 3
            ],
            [
                "All Missing",
                "Double Entry",
                "Triple Entry",
                "More Than 3 Entry"
            ],
            default=""
        )

        df.replace({0: 'Missing'}, inplace=True)
        df.drop(columns=['A4_max'], inplace=True)
        # print('A1D1')
        # print(df)

        # ---------------------------- check data correction ----------------------------
        # -------------------------------------------------------------------------------
        data_correction_df = get_data_correction()
        data_correction_df = data_correction_df[data_correction_df['Form'] == 'A1']

        # print(data_correction_df)
        df = df[
            ~df['A1'].isin(data_correction_df['Participant ID'])
        ]

        # Keep rows where the last element does NOT start with 7 or 9
        df = df[
            ~df['A1'].str.split('-').str[-1].str.startswith(('7', '9'))
        ]
        # print(df[~df['A1'].isin(data_correction_df['Participant ID'])])
        # print(df)

        # =========================
        # Split data
        # =========================
        df_sr = df[df['A1'].str.contains('SR', na=False)]
        df_pp = df[df['A1'].str.contains('PP', na=False)]

        # =========================
        # Save Excel
        # =========================
        filename = f"A1D1_DataQuality_{datetime.now().strftime('%d%m%Y')}.xlsx"
        output_path = os.path.join(tempfile.gettempdir(), filename)

        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            df.rename(columns=a1_section_mapping).to_excel(writer, sheet_name='ALL', index=False)
            df_pp.rename(columns=a1_section_mapping).to_excel(writer, sheet_name='PP', index=False)
            df_sr.rename(columns=a1_section_mapping).to_excel(writer, sheet_name='SR', index=False)

        # =========================
        # Return (limit preview size)
        # =========================
        return (
            f"✅ Processing complete | Issues found - ALL: {len(df)}, PP: {len(df_pp)}, SR: {len(df_sr)}",
            output_path,
            df.reset_index().rename(columns={"index": "No."}).style.map(lambda x: "background-color: #ffe6e6" if x == "Missing" else ""),
            df_pp.reset_index().rename(columns={"index": "No."}).style.map(lambda x: "background-color: #ffe6e6" if x == "Missing" else ""),
            df_sr.reset_index().rename(columns={"index": "No."}).style.map(lambda x: "background-color: #ffe6e6" if x == "Missing" else ""),
            len(df),
            len(df_pp),
            len(df_sr)
        )

    except Exception as e:
        return f"❌ Error: {str(e)}", None, None, None, None, None, None, None


# =========================
# =========================
# Main Processing Function A4D4
# =========================
# =========================
def process_files_a4d4(a4_file, d4_file):
    a5_section_mapping = {
        'A1'    : 'ID',
        'A1manual' : 'ID Manual',
        'A4/1'  : 'B.Inclusion Criteria-A4/1',
        'A4/2'  : 'C.Demographic Information-A4/2',
        'A4/3'  : 'D.Animal Contact Information-A4/3',
        'A4/4'  : 'E.Mosquito Activity Information-A4/4',
        'A4/5'  : 'F.Information from the Child Traveling, Daycare / Schools and Sibling-A4/5',
        'A4/6'  : 'G.Child Vaccination Record-A4/6',
        'A4/7'  : 'H.Child Kuntha Bopha Hospitalization-A4/7',
        'A4/8'  : 'I.Child Medical History-A4/8',
        'A4/9'  : 'J.Physical Examination​ and Neurological Assessment-A4/9',
        'A4/10'  : 'K.Initial Prescription-A4/10',
        'A4/11'  : 'L.Prescription During the Course of Disease/Hospitalization-A4/11',
        'A4/12'  : 'M.Discharge Status-A4/12',
        'ID'    : 'A1'
    }
    if a4_file is None or d4_file is None:
        return "❌ Please upload both A1 and D1 files", None, None, None, None, None, None, None

    try:
        # =========================
        # Load files (HF compatible)
        # =========================
        a4_df = pd.read_csv(a4_file, low_memory=False)
        d4_df = pd.read_csv(d4_file, low_memory=False)

        # =========================
        # A1 processing
        # =========================
        a4_df['SubmissionDate'] = pd.to_datetime(a4_df['SubmissionDate'], errors='coerce')
        a4_df.sort_values(by='SubmissionDate', ascending=False, inplace=True)

        A4_cols = [f'A4/{i}' for i in range(1,13)]
        A_section = ['SubmissionDate', 'A1', 'A1manual'] + A4_cols
        a4_df = a4_df[A_section]

        a4_df['A1'] = a4_df['A1'].fillna(a4_df['A1manual'])
        a4_df[A4_cols] = a4_df[A4_cols].apply(pd.to_numeric, errors='coerce')

        # =========================
        # D1 processing
        # =========================
        d4_df['A1'] = d4_df['A1'].fillna(d4_df['A1manual'])

        useful_column = ['SubmissionDate','A1', 'A1manual', 'SubmitterID', 'SubmitterName', 'Edits']
        d4_df = d4_df[useful_column]

        # =========================
        # Main Logic
        # =========================
        result_dict = {}

        for id in d4_df['A1'].dropna().unique():
            tmp_df = a4_df[a4_df['A1'] == id]

            if tmp_df.empty:
                continue

            condition_dict = {
                col: int(tmp_df[col].sum(skipna=True))
                for col in A4_cols
            }

            condition_dict['Record in A4'] = tmp_df.shape[0]

            if not all(condition_dict[col] == 1 for col in A4_cols):
                result_dict[id] = condition_dict

        # =========================
        # Convert to DataFrame
        # =========================
        df = pd.DataFrame.from_dict(result_dict, orient='index').reset_index()
        df = df.rename(columns={'index':'A1'})

        if df.empty:
            return "✅ NO ISSUES FOUND", None, None, None, None, None, None, None

        # =========================
        # Add remarks
        # =========================
        df['A4_max'] = df[A4_cols].max(axis=1)

        df['Remark'] = np.select(
            [
                df['A4_max'] == 0,
                df['A4_max'] == 2,
                df['A4_max'] == 3,
                df['A4_max'] > 3
            ],
            [
                "All Missing",
                "Double Entry",
                "Triple Entry",
                "More Than 3 Entry"
            ],
            default=""
        )

        df.replace({0: 'Missing'}, inplace=True)
        df.drop(columns=['A4_max'], inplace=True)
        # print('A4D4')
        # print(df)

        # =========================
        # Split data
        # =========================
        df_sr = df[df['A1'].str.contains('SR', na=False)]
        df_pp = df[df['A1'].str.contains('PP', na=False)]

        # =========================
        # Save Excel
        # =========================
        filename = f"A4D4_DataQuality_{datetime.now().strftime('%d%m%Y')}.xlsx"
        output_path = os.path.join(tempfile.gettempdir(), filename)

        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            df.rename(columns=a5_section_mapping).to_excel(writer, sheet_name='ALL', index=False)
            df_pp.rename(columns=a5_section_mapping).to_excel(writer, sheet_name='PP', index=False)
            df_sr.rename(columns=a5_section_mapping).to_excel(writer, sheet_name='SR', index=False)

        # =========================
        # Return (limit preview size)
        # =========================
        return (
            f"✅ Processing complete | Issues found: ALL {len(df)}, PP {len(df_pp)}, SR {len(df_sr)}",
            output_path,
            df.reset_index().rename(columns={"index": "No."}).style.map(lambda x: "background-color: #ffe6e6" if x == "Missing" else ""),
            df_pp.reset_index().rename(columns={"index": "No."}).style.map(lambda x: "background-color: #ffe6e6" if x == "Missing" else ""),
            df_sr.reset_index().rename(columns={"index": "No."}).style.map(lambda x: "background-color: #ffe6e6" if x == "Missing" else ""),
            len(df),
            len(df_pp),
            len(df_sr)
        )

    except Exception as e:
        return f"❌ Error: {str(e)}", None, None, None, None, None, None, None

# =========================
# =========================
# Main Processing Function A5D5
# =========================
# =========================
def process_files_a5d5(a5_file, d5_file):
    a5_section_mapping = {
        'A1'    : 'ID',
        'A1manual' : 'ID Manual',
        'A4/1'  : 'B.Health Assessment-A4/1',
        'A4/2'  : 'C.LIVERPOOL Score-A4/2',
        'ID'    : 'A1'
    }
    if a5_file is None or d5_file is None:
        return "❌ Please upload both A1 and D1 files", None, None, None, None, None, None, None

    try:
        # =========================
        # Load files (HF compatible)
        # =========================
        a5_df = pd.read_csv(a5_file, low_memory=False)
        d5_df = pd.read_csv(d5_file, low_memory=False)

        # =========================
        # A1 processing
        # =========================
        a5_df['SubmissionDate'] = pd.to_datetime(a5_df['SubmissionDate'], errors='coerce')
        a5_df.sort_values(by='SubmissionDate', ascending=False, inplace=True)

        A4_cols = [f'A4/{i}' for i in range(1,3)]
        A_section = ['SubmissionDate', 'A1', 'A1manual'] + A4_cols
        a5_df = a5_df[A_section]

        a5_df['A1'] = a5_df['A1'].fillna(a5_df['A1manual'])
        a5_df[A4_cols] = a5_df[A4_cols].apply(pd.to_numeric, errors='coerce')

        # =========================
        # D1 processing
        # =========================
        d5_df['A1'] = d5_df['A1'].fillna(d5_df['A1manual'])

        useful_column = ['SubmissionDate','A1', 'A1manual', 'SubmitterID', 'SubmitterName', 'Edits']
        d5_df = d5_df[useful_column]

        # =========================
        # Main Logic
        # =========================
        result_dict = {}

        for id in d5_df['A1'].dropna().unique():
            tmp_df = a5_df[a5_df['A1'] == id]

            if tmp_df.empty:
                continue

            condition_dict = {
                col: int(tmp_df[col].sum(skipna=True))
                for col in A4_cols
            }

            condition_dict['Record in A5'] = tmp_df.shape[0]

            if not all(condition_dict[col] == 1 for col in A4_cols):
                result_dict[id] = condition_dict

        # =========================
        # Convert to DataFrame
        # =========================
        df = pd.DataFrame.from_dict(result_dict, orient='index').reset_index()
        df = df.rename(columns={'index':'A1'})

        if df.empty:
            return "✅ NO ISSUES FOUND", None, None, None, None, None, None, None

        # =========================
        # Add remarks
        # =========================
        df['A4_max'] = df[A4_cols].max(axis=1)

        df['Remark'] = np.select(
            [
                df['A4_max'] == 0,
                df['A4_max'] == 2,
                df['A4_max'] == 3,
                df['A4_max'] > 3
            ],
            [
                "All Missing",
                "Double Entry",
                "Triple Entry",
                "More Than 3 Entry"
            ],
            default=""
        )

        df.replace({0: 'Missing'}, inplace=True)
        df.drop(columns=['A4_max'], inplace=True)
        # print('A5D5')
        # print(df)

        # =========================
        # Split data
        # =========================
        df_sr = df[df['A1'].str.contains('SR', na=False)]
        df_pp = df[df['A1'].str.contains('PP', na=False)]

        # =========================
        # Save Excel
        # =========================
        filename = f"A5D5_DataQuality_{datetime.now().strftime('%d%m%Y')}.xlsx"
        output_path = os.path.join(tempfile.gettempdir(), filename)

        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            df.rename(columns=a5_section_mapping).to_excel(writer, sheet_name='ALL', index=False)
            df_pp.rename(columns=a5_section_mapping).to_excel(writer, sheet_name='PP', index=False)
            df_sr.rename(columns=a5_section_mapping).to_excel(writer, sheet_name='SR', index=False)

        # =========================
        # Return (limit preview size)
        # =========================
        return (
            f"✅ Processing complete | Issues found: ALL {len(df)}, PP {len(df_pp)}, SR {len(df_sr)}",
            output_path,
            df.reset_index().rename(columns={"index": "No."}).style.map(lambda x: "background-color: #ffe6e6" if x == "Missing" else ""),
            df_pp.reset_index().rename(columns={"index": "No."}).style.map(lambda x: "background-color: #ffe6e6" if x == "Missing" else ""),
            df_sr.reset_index().rename(columns={"index": "No."}).style.map(lambda x: "background-color: #ffe6e6" if x == "Missing" else ""),
            len(df),
            len(df_pp),
            len(df_sr)
        )

    except Exception as e:
        return f"❌ Error: {str(e)}", None, None, None, None, None, None, None



# =========================
# =========================
# Gradio UI
# =========================
# =========================
with gr.Blocks(title="Data Validation Tool") as app:

    gr.Markdown("# 📊 DEMELE JEV Data Checking and Validation Tool")
    with gr.Tabs():

        # =========================
        # TAB 1: A1-D1
        # =========================
        with gr.Tab("A1-D1"):

            gr.Markdown("## A1-D1 Validation")

            with gr.Row():
                a1_input = gr.File(label="Upload A1_Questionnaire_Cohort 1_Baseline CSV")
                d1_input = gr.File(label="Upload D1_Screening Form_Cohort 1_Baseline CSV")

            run_btn_a1 = gr.Button("🚀 Run A1-D1 Check", variant="primary")

            with gr.Row():
                status_output_a1 = gr.Textbox(label="Status", scale=2)
                file_output_a1 = gr.File(label="Download Result Excel", scale=1)
            
            # add Card Vis
            with gr.Row():
                label_all = gr.Label(label="ALL")
                label_pp = gr.Label(label="PP")
                label_sr = gr.Label(label="SR")
            table_output_a1 = gr.Dataframe(label="A1D1 Missing Information ALL")
            table_output1_a1 = gr.Dataframe(label="A1D1 Missing Information PP")
            table_output2_a1 = gr.Dataframe(label="A1D1 Missing Information SR")

            run_btn_a1.click(
                fn=process_files_a1d1,
                inputs=[a1_input, d1_input],
                outputs=[
                    status_output_a1,
                    file_output_a1,
                    table_output_a1,
                    table_output1_a1,
                    table_output2_a1,
                    label_all,
                    label_pp,
                    label_sr
                ]
            )

        # =========================
        # TAB 2: A4-D4
        # =========================
        with gr.Tab("A4-D4"):

            gr.Markdown("## A4-D4 Validation")

            with gr.Row():
                a4_input = gr.File(label="Upload A4_Questionnaire_Cohort 2_Baseline CSV")
                d4_input = gr.File(label="Upload D4_Screening Form_Cohort 2_Baseline CSV")

            run_btn_a4 = gr.Button("🚀 Run A4-D4 Check", variant="primary")

            with gr.Row():
                status_output_a4 = gr.Textbox(label="Status", scale=2)
                file_output_a4 = gr.File(label="Download Result Excel", scale=1)

            # add Card Vis
            with gr.Row():
                label_all = gr.Label(label="ALL")
                label_pp = gr.Label(label="PP")
                label_sr = gr.Label(label="SR")
            table_output_a4 = gr.Dataframe(label="A4D4 Missing Information ALL")
            table_output1_a4 = gr.Dataframe(label="A4D4 Missing Information PP")
            table_output2_a4 = gr.Dataframe(label="A4D4 Missing Information SR")

            run_btn_a4.click(
                fn=process_files_a4d4,   # 👈 create this function
                inputs=[a4_input, d4_input],
                outputs=[
                    status_output_a4,
                    file_output_a4,
                    table_output_a4,
                    table_output1_a4,
                    table_output2_a4,
                    label_all,
                    label_pp,
                    label_sr
                ]
            )

        # =========================
        # TAB 3: A5-D5
        # =========================
        with gr.Tab("A5-D5"):

            gr.Markdown("## A5-D5 Validation")

            with gr.Row():
                a5_input = gr.File(label="Upload A5_Questionnaire_Cohort2_M1 CSV")
                d5_input = gr.File(label="Upload D5_Screening Form_Cohort 2_M1 CSV")

            run_btn_a5 = gr.Button("🚀 Run A5-D5 Check", variant="primary")

            with gr.Row():
                status_output_a5 = gr.Textbox(label="Status", scale=2)
                file_output_a5 = gr.File(label="Download Result Excel", scale=1)

            # add Card Vis
            with gr.Row():
                label_all = gr.Label(label="ALL")
                label_pp = gr.Label(label="PP")
                label_sr = gr.Label(label="SR")
            table_output_a5 = gr.Dataframe(label="A5D5 Missing Information ALL")
            table_output1_a5 = gr.Dataframe(label="A5D5 Missing Information PP")
            table_output2_a5 = gr.Dataframe(label="A5D5 Missing Information SR")

            run_btn_a5.click(
                fn=process_files_a5d5,   # 👈 create this function
                inputs=[a5_input, d5_input],
                outputs=[
                    status_output_a5,
                    file_output_a5,
                    table_output_a5,
                    table_output1_a5,
                    table_output2_a5,
                    label_all,
                    label_pp,
                    label_sr
                ]
            )
        with gr.Tab("Cleaning Pipeline"):
            gr.Markdown("## Data Cleaning Pipeline")
            gr.Markdown("This section is under development. Please check back later for updates on the data cleaning pipeline.")
        
        with gr.Tab("Preparation for API"):
            gr.Markdown("## Dataset Preparation for API")
            gr.Markdown("This section is under development. Please check back later for updates on dataset preparation for analysis.")
        
        with gr.Tab("Visualization"):
            gr.Markdown("## Data Information")
            gr.Markdown("This section is under development. Please check back later for updates on dataset preparation for analysis.")

# Launch (HF compatible)
app.queue()
# app.launch(server_name="0.0.0.0", server_port=7860, ssr_mode=False)
app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
